# 04c（US）置换检验与序列相关稳健性（Permutation / HAC / Block Bootstrap）

目的：避免把多重比较噪声或序列相关处理当成“干支有效”。

输入：
- `data/clean_us/market_ganzhi_{ts_code}.csv.gz`

输出（写入 `data/clean_us/robustness/`）：
- `perm_global.csv`：全局置换检验（raw；`stem/branch/(可选 ganzhi_day)` × `ret_1d/up`）
- `perm_global_controls_resid.csv`：先回归掉 `weekday/month/year` 后的残差置换（更保守）
- `hac_sensitivity_controls_{ts_code}_stem_ret_1d.csv`：HAC `maxlags` 敏感性（stem×ret_1d）
- `block_bootstrap_controls_resid_{ts_code}_stem_ret_1d.csv`：block bootstrap（controls 残差；stem×ret_1d）


In [1]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns



from matplotlib import font_manager, rcParams


def set_chinese_font() -> None:
    # Best-effort Chinese font setup for Matplotlib.
    # - Tries common Linux CJK fonts
    # - On WSL, also tries registering Windows fonts under /mnt/c/Windows/Fonts

    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK TC',
        'Noto Sans CJK JP',
        'Source Han Sans SC',
        'WenQuanYi Micro Hei',
        'Microsoft YaHei',
        'SimHei',
        'SimSun',
        'Arial Unicode MS',
    ]

    # WSL fallback: register Windows fonts if available
    try:
        from pathlib import Path as _Path

        win_dir = _Path('/mnt/c/Windows/Fonts')
        win_files = [
            win_dir / 'msyh.ttc',
            win_dir / 'msyhbd.ttc',
            win_dir / 'simhei.ttf',
            win_dir / 'simsun.ttc',
            win_dir / 'arialuni.ttf',
        ]

        extra_names: list[str] = []
        addfont = getattr(font_manager.fontManager, 'addfont', None)
        for fp in win_files:
            try:
                if fp.is_file():
                    if callable(addfont):
                        addfont(str(fp))
                    extra_names.append(font_manager.FontProperties(fname=str(fp)).get_name())
            except Exception:
                continue

        candidates = [n for n in extra_names if n] + candidates
    except Exception:
        pass

    for name in candidates:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.sans-serif'] = [name]
            rcParams['axes.unicode_minus'] = False
            print(f'Using Chinese font: {name}')
            return
        except Exception:
            continue

    rcParams['axes.unicode_minus'] = False
    print(
        'Warning: no Chinese font found; Chinese labels may not render. '
        'If you are on WSL, ensure Windows fonts are accessible under /mnt/c/Windows/Fonts; '
        'or install Noto CJK fonts in Linux (e.g., fonts-noto-cjk).'
    )


set_chinese_font()

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean_us'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['spx', 'ndq', 'ndx', 'dji']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks_US 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out




Using Chinese font: Microsoft YaHei
PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支


## 0) HAC 敏感性所需的边际效应工具函数


In [2]:
@dataclass
class ControlModelSpec:
    group_col: str
    target: str  # 'up' or 'ret_1d'
    formula: str


def fit_control_model(df: pd.DataFrame, spec: ControlModelSpec):
    if spec.target == 'up':
        model = smf.glm(formula=spec.formula, data=df, family=sm.families.Binomial())
        return model.fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS}, maxiter=200, disp=0)

    if spec.target == 'ret_1d':
        model = smf.ols(formula=spec.formula, data=df)
        return model.fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})

    raise ValueError(f'Unknown target: {spec.target}')


def extract_group_terms(res, group_col: str) -> pd.DataFrame:
    params = res.params
    pvals = res.pvalues

    rows = []
    for name in params.index:
        if name.startswith(f'C({group_col})['):
            rows.append(
                {
                    'term': name,
                    'coef': float(params[name]),
                    'p_value': float(pvals[name]) if np.isfinite(pvals[name]) else np.nan,
                }
            )

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    p = out['p_value'].to_numpy(dtype=float)
    ok = np.isfinite(p)
    q = np.full_like(p, np.nan, dtype=float)
    if ok.sum() > 0:
        q[ok] = fdr_bh(p[ok])
    out['q_value'] = q

    out['group_col'] = group_col
    out['group_value'] = out['term'].str.extract(r'\[T\.(.*)\]')[0]

    return out.sort_values(['q_value', 'p_value'])


def marginal_mean_by_group(res, df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    # Marginal mean: set group_col to each value and average predictions over observed controls
    values = pd.Series(df[group_col].dropna().unique()).astype(str).sort_values().tolist()

    if isinstance(df[group_col].dtype, pd.CategoricalDtype):
        values = [str(v) for v in df[group_col].dtype.categories if v in set(df[group_col].dropna())]

    base_pred = res.predict(df)
    base_mean = float(np.nanmean(base_pred))

    records = []
    for v in values:
        tmp = df.copy()
        if isinstance(df[group_col].dtype, pd.CategoricalDtype):
            tmp[group_col] = pd.Categorical(
                [v] * len(tmp),
                categories=df[group_col].dtype.categories,
                ordered=df[group_col].dtype.ordered,
            )
        else:
            tmp[group_col] = v
        pred = res.predict(tmp)
        mu = float(np.nanmean(pred))
        records.append(
            {
                'group_col': group_col,
                'group_value': v,
                'marginal_mean': mu,
                'marginal_effect_vs_all': mu - base_mean,
            }
        )

    out = pd.DataFrame.from_records(records)
    out['marginal_mean_all'] = base_mean
    return out


def marginal_effect_tests_vs_all(
    res,
    df: pd.DataFrame,
    group_col: str,
    target: str,
    group_values: list[str],
) -> pd.DataFrame:
    values = [str(v) for v in group_values]

    if target == 'ret_1d':
        design_info = res.model.data.design_info
        x_base = res.model.exog
        x_base_mean = np.nanmean(x_base, axis=0)

        records = []
        for v in values:
            tmp = df.copy()
            if isinstance(df[group_col].dtype, pd.CategoricalDtype):
                tmp[group_col] = pd.Categorical(
                    [v] * len(tmp),
                    categories=df[group_col].dtype.categories,
                    ordered=df[group_col].dtype.ordered,
                )
            else:
                tmp[group_col] = v

            x_v = build_design_matrices([design_info], tmp, return_type='dataframe')[0].to_numpy()
            l = np.nanmean(x_v, axis=0) - x_base_mean
            t = res.t_test(l)

            se = float(np.asarray(t.sd).ravel()[0])
            z = float(np.asarray(t.tvalue).ravel()[0])
            p = float(np.asarray(t.pvalue).ravel()[0])
            records.append({'group_value': v, 'se_effect': se, 'z_effect': z, 'p_value_effect': p})

        out = pd.DataFrame.from_records(records)
        p = out['p_value_effect'].to_numpy(dtype=float)
        ok = np.isfinite(p)
        q = np.full_like(p, np.nan, dtype=float)
        if ok.sum() > 0:
            q[ok] = fdr_bh(p[ok])
        out['q_value_effect'] = q
        return out

    if target == 'up':
        beta = res.params.to_numpy()
        cov = res.cov_params().to_numpy()

        design_info = res.model.data.design_info
        x_base = res.model.exog
        eta_base = x_base @ beta
        mu_base = res.model.family.link.inverse(eta_base)
        base_mean = float(np.nanmean(mu_base))
        w_base = res.model.family.link.inverse_deriv(eta_base)
        g_base = np.nanmean(w_base[:, None] * x_base, axis=0)

        records = []
        for v in values:
            tmp = df.copy()
            if isinstance(df[group_col].dtype, pd.CategoricalDtype):
                tmp[group_col] = pd.Categorical(
                    [v] * len(tmp),
                    categories=df[group_col].dtype.categories,
                    ordered=df[group_col].dtype.ordered,
                )
            else:
                tmp[group_col] = v

            x_v = build_design_matrices([design_info], tmp, return_type='dataframe')[0].to_numpy()
            eta_v = x_v @ beta
            mu_v = res.model.family.link.inverse(eta_v)
            mean_v = float(np.nanmean(mu_v))

            w_v = res.model.family.link.inverse_deriv(eta_v)
            g_v = np.nanmean(w_v[:, None] * x_v, axis=0)

            effect = mean_v - base_mean
            g_diff = g_v - g_base
            var = float(g_diff @ cov @ g_diff)

            if not np.isfinite(var) or var <= 0:
                se = np.nan
                z = np.nan
                p = np.nan
            else:
                se = float(np.sqrt(var))
                z = float(effect / se)
                p = float(2.0 * stats.norm.sf(abs(z)))

            records.append({'group_value': v, 'se_effect': se, 'z_effect': z, 'p_value_effect': p})

        out = pd.DataFrame.from_records(records)
        p = out['p_value_effect'].to_numpy(dtype=float)
        ok = np.isfinite(p)
        q = np.full_like(p, np.nan, dtype=float)
        if ok.sum() > 0:
            q[ok] = fdr_bh(p[ok])
        out['q_value_effect'] = q
        return out

    raise ValueError(f'Unknown target: {target}')




## 1) HAC maxlags 敏感性（stem × ret_1d）


In [3]:
def run_hac_sensitivity_stem_ret_1d(ts_code: str, maxlags_list: list[int]) -> pd.DataFrame:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'stem', 'weekday', 'month', 'year'])

    if not isinstance(df['stem'].dtype, pd.CategoricalDtype):
        df['stem'] = df['stem'].astype('category')

    records: list[pd.DataFrame] = []
    for maxlags in maxlags_list:
        res = smf.ols(
            formula='ret_1d ~ C(stem) + C(weekday) + C(month) + C(year)',
            data=df,
        ).fit(cov_type='HAC', cov_kwds={'maxlags': int(maxlags)})

        mm_tbl = marginal_mean_by_group(res, df, 'stem')
        eff_tbl = marginal_effect_tests_vs_all(res, df, 'stem', 'ret_1d', mm_tbl['group_value'].tolist())

        out = mm_tbl.merge(eff_tbl, on='group_value', how='left')
        out.insert(0, 'maxlags', int(maxlags))
        out.insert(0, 'ts_code', ts_code)

        keep = [
            'ts_code',
            'maxlags',
            'group_value',
            'marginal_effect_vs_all',
            'se_effect',
            'z_effect',
            'p_value_effect',
            'q_value_effect',
        ]
        records.append(out[keep].copy())

    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


for ts_code in TS_CODES:
    try:
        sens = run_hac_sensitivity_stem_ret_1d(ts_code, HAC_MAXLAGS_SWEEP)
    except Exception as exc:
        print('[HAC sensitivity] failed:', ts_code, exc)
        continue

    out_path = ROBUST_DIR / f'hac_sensitivity_controls_{ts_code}_stem_ret_1d.csv'
    sens.to_csv(out_path, index=False)
    print('saved:', out_path)

    view = sens[sens['group_value'].astype(str) == '丙'].sort_values('maxlags')
    if not view.empty:
        print('[HAC sensitivity] stem=丙')
        print(view[['maxlags', 'marginal_effect_vs_all', 'p_value_effect', 'q_value_effect']].to_string(index=False))




saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\hac_sensitivity_controls_spx_stem_ret_1d.csv
[HAC sensitivity] stem=丙
 maxlags  marginal_effect_vs_all  p_value_effect  q_value_effect
       0                0.000243        0.617486        0.971947
       1                0.000243        0.621852        0.971952
       3                0.000243        0.624793        0.972479
       5                0.000243        0.627061        0.972665
      10                0.000243        0.633310        0.972094
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\hac_sensitivity_controls_ndq_stem_ret_1d.csv
[HAC sensitivity] stem=丙
 maxlags  marginal_effect_vs_all  p_value_effect  q_value_effect
       0                0.000291        0.613688        0.998135
       1                0.000291        0.617077        0.998158
       3                0.000291        0.620286        0.998141
       5                0.000291        0.622496        0.998146
      10                0.000

## 2) 全局置换检验（raw series）


In [4]:
def perm_global_test(df: pd.DataFrame, group_col: str, target: str, n_perm: int, seed: int) -> dict:
    x = df[target].to_numpy(dtype=float)
    overall = float(np.nanmean(x))

    cat = pd.Categorical(df[group_col])
    codes = cat.codes
    k = int(cat.categories.size)

    ok = codes >= 0
    codes = codes[ok]
    x = x[ok]

    def stat(c: np.ndarray) -> float:
        cnt = np.bincount(c, minlength=k)
        s = np.bincount(c, weights=x, minlength=k)
        mu = np.divide(s, cnt, out=np.full(k, np.nan), where=cnt > 0)
        eff = mu - overall
        return float(np.nanmax(np.abs(eff)))

    t_obs = stat(codes)

    rng = np.random.default_rng(seed)
    t_perm = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        t_perm[i] = stat(rng.permutation(codes))

    p_emp = (1.0 + float((t_perm >= t_obs).sum())) / (n_perm + 1.0)

    return {
        'group_col': group_col,
        'target': target,
        'k_groups': k,
        't_obs': t_obs,
        'p_empirical': p_emp,
        't_perm_mean': float(t_perm.mean()),
        't_perm_p95': float(np.quantile(t_perm, 0.95)),
        't_perm_p99': float(np.quantile(t_perm, 0.99)),
    }


records = []
for ts_code in TS_CODES:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'up'])

    cols = ['stem', 'branch'] + (['ganzhi_day'] if RUN_GANZHI_DAY_MODELS else [])
    for group_col in cols:
        for target in ['ret_1d', 'up']:
            rec = perm_global_test(df, group_col, target, n_perm=N_PERM, seed=RANDOM_SEED)
            rec['ts_code'] = ts_code
            rec['n'] = int(len(df))
            records.append(rec)
            print(ts_code, group_col, target, 'p_empirical=', rec['p_empirical'])

perm_df = pd.DataFrame.from_records(records)
out_path = ROBUST_DIR / 'perm_global.csv'
perm_df.to_csv(out_path, index=False)
print('saved:', out_path)
perm_df


# ------------------------------
# Permutation on controls-only residuals (stem × ret_resid)
# ------------------------------



spx stem ret_1d p_empirical= 0.9370629370629371
spx stem up p_empirical= 0.32967032967032966
spx branch ret_1d p_empirical= 0.5034965034965035
spx branch up p_empirical= 0.43156843156843155
spx ganzhi_day ret_1d p_empirical= 0.8471528471528471
spx ganzhi_day up p_empirical= 0.5614385614385614
ndq stem ret_1d p_empirical= 0.8401598401598401
ndq stem up p_empirical= 0.8641358641358642
ndq branch ret_1d p_empirical= 0.21578421578421578
ndq branch up p_empirical= 0.22677322677322678
ndq ganzhi_day ret_1d p_empirical= 0.6183816183816184
ndq ganzhi_day up p_empirical= 0.6223776223776224
ndx stem ret_1d p_empirical= 0.7812187812187812
ndx stem up p_empirical= 0.975024975024975
ndx branch ret_1d p_empirical= 0.2177822177822178
ndx branch up p_empirical= 0.3156843156843157
ndx ganzhi_day ret_1d p_empirical= 0.6913086913086913
ndx ganzhi_day up p_empirical= 0.8361638361638362
dji stem ret_1d p_empirical= 0.939060939060939
dji stem up p_empirical= 0.7792207792207793
dji branch ret_1d p_empirical=

,group_col,target,k_groups,t_obs,p_empirical,t_perm_mean,t_perm_p95,t_perm_p99,ts_code,n
0,stem,ret_1d,10,0.000582,0.937063,0.000958,0.001452,0.001709,spx,4053
1,stem,up,10,0.048086,0.329670,0.044213,0.067507,0.079842,spx,4053
2,branch,ret_1d,12,0.001082,0.503497,0.001112,0.001666,0.001918,spx,4053
3,branch,up,12,0.051706,0.431568,0.050768,0.074532,0.088180,spx,4053
4,ganzhi_day,ret_1d,60,0.002855,0.847153,0.003472,0.004590,0.005477,spx,4053
5,ganzhi_day,up,60,0.149143,0.561439,0.153385,0.199760,0.227668,spx,4053
6,stem,ret_1d,10,0.000838,0.840160,0.001135,0.001647,0.001909,ndq,4054
7,stem,up,10,0.031220,0.864136,0.043521,0.065011,0.075799,ndq,4054
8,branch,ret_1d,12,0.001563,0.215784,0.001305,0.001945,0.002211,ndq,4054
9,branch,up,12,0.059927,0.226773,0.050314,0.073351,0.085569,ndq,4054


## 3) 残差置换检验（controls-only residual；更保守）


In [5]:
def add_controls_only_residual(df: pd.DataFrame) -> pd.DataFrame:
    tmp = df.copy()
    res = smf.ols(
        formula='ret_1d ~ C(weekday) + C(month) + C(year)',
        data=tmp,
    ).fit()
    tmp['ret_resid'] = res.resid
    return tmp


resid_records = []
for ts_code in TS_CODES:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'stem', 'weekday', 'month', 'year'])

    df = add_controls_only_residual(df)

    rec = perm_global_test(df, group_col='stem', target='ret_resid', n_perm=N_PERM_RESID, seed=RANDOM_SEED + 1)
    rec['ts_code'] = ts_code
    rec['n'] = int(len(df))
    rec['n_perm'] = int(N_PERM_RESID)
    rec['seed'] = int(RANDOM_SEED + 1)
    resid_records.append(rec)
    print(ts_code, 'stem', 'ret_resid', 'p_empirical=', rec['p_empirical'])

perm_resid_df = pd.DataFrame.from_records(resid_records)
out_path = ROBUST_DIR / 'perm_global_controls_resid.csv'
perm_resid_df.to_csv(out_path, index=False)
print('saved:', out_path)
perm_resid_df

# ------------------------------
# Block bootstrap on controls-only residuals (stem × ret_resid)
# ------------------------------



spx stem ret_resid p_empirical= 0.9380309845077461
ndq stem ret_resid p_empirical= 0.8325837081459271
ndx stem ret_resid p_empirical= 0.8140929535232384
dji stem ret_resid p_empirical= 0.936031984007996
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\perm_global_controls_resid.csv


,group_col,target,k_groups,t_obs,p_empirical,t_perm_mean,t_perm_p95,t_perm_p99,ts_code,n,n_perm,seed
0,stem,ret_resid,10,0.000565,0.938031,0.000962,0.001477,0.001699,spx,4053,2000,20260214
1,stem,ret_resid,10,0.000816,0.832584,0.001132,0.001707,0.001981,ndq,4054,2000,20260214
2,stem,ret_resid,10,0.000862,0.814093,0.001153,0.001741,0.002026,ndx,4053,2000,20260214
3,stem,ret_resid,10,0.000560,0.936032,0.000922,0.001415,0.001640,dji,4053,2000,20260214


## 4) Block bootstrap（controls 残差；序列相关稳健性）


In [6]:
def circular_block_bootstrap_indices(n: int, block_len: int, rng: np.random.Generator) -> np.ndarray:
    if n <= 0:
        return np.array([], dtype=int)
    block_len = int(max(1, block_len))
    n_blocks = int(np.ceil(n / block_len))
    starts = rng.integers(0, n, size=n_blocks)
    offsets = np.arange(block_len)
    idx = (starts[:, None] + offsets[None, :]) % n
    return idx.ravel()[:n]


def block_bootstrap_stem_effects(df: pd.DataFrame, n_boot: int, block_len: int, seed: int) -> pd.DataFrame:
    tmp = df.sort_values('date').copy()

    cat = pd.Categorical(tmp['stem'], categories=STEMS_ORDER, ordered=True)
    codes = cat.codes
    ok = codes >= 0
    codes = codes[ok]
    resid = tmp.loc[ok, 'ret_resid'].to_numpy(dtype=float)

    k = int(len(STEMS_ORDER))
    overall = float(np.nanmean(resid))

    cnt = np.bincount(codes, minlength=k)
    s = np.bincount(codes, weights=resid, minlength=k)
    mu = np.divide(s, cnt, out=np.full(k, np.nan), where=cnt > 0)
    eff_obs = mu - overall

    rng = np.random.default_rng(int(seed))
    boot = np.empty((int(n_boot), k), dtype=float)

    n = int(resid.size)
    for b in range(int(n_boot)):
        idx = circular_block_bootstrap_indices(n, block_len=block_len, rng=rng)
        x = resid[idx]
        c = codes[idx]
        ov = float(np.nanmean(x))

        cnt_b = np.bincount(c, minlength=k)
        s_b = np.bincount(c, weights=x, minlength=k)
        mu_b = np.divide(s_b, cnt_b, out=np.full(k, np.nan), where=cnt_b > 0)
        boot[b, :] = mu_b - ov

    records = []
    for i, stem in enumerate(STEMS_ORDER):
        vals = boot[:, i]
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            boot_mean = np.nan
            ci_low = np.nan
            ci_high = np.nan
            p_boot = np.nan
        else:
            boot_mean = float(np.mean(vals))
            ci_low = float(np.quantile(vals, 0.025))
            ci_high = float(np.quantile(vals, 0.975))
            p_boot = float(2.0 * min(np.mean(vals <= 0.0), np.mean(vals >= 0.0)))

        records.append(
            {
                'group_value': stem,
                'effect_obs': float(eff_obs[i]) if np.isfinite(eff_obs[i]) else np.nan,
                'boot_mean': boot_mean,
                'boot_ci_low': ci_low,
                'boot_ci_high': ci_high,
                'p_boot_two_sided': p_boot,
                'n_boot': int(n_boot),
                'block_len': int(block_len),
                'n_days': int(n),
            }
        )

    out = pd.DataFrame.from_records(records)
    out.insert(0, 'ts_code', str(tmp['ts_code'].iloc[0]) if 'ts_code' in tmp.columns and len(tmp) else '')
    return out


for ts_code in TS_CODES:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)
    df = df.dropna(subset=['ret_1d', 'stem', 'weekday', 'month', 'year'])
    df = add_controls_only_residual(df)

    seed = int(RANDOM_SEED + sum(ord(ch) for ch in str(ts_code)))
    boot_df = block_bootstrap_stem_effects(df.assign(ts_code=ts_code), n_boot=N_BOOT, block_len=BOOT_BLOCK_LEN, seed=seed)
    out_path = ROBUST_DIR / f'block_bootstrap_controls_resid_{ts_code}_stem_ret_1d.csv'
    boot_df.to_csv(out_path, index=False)
    print('saved:', out_path)

    view = boot_df[boot_df['group_value'] == '丙']
    if not view.empty:
        print('[block bootstrap] stem=丙')
        print(view[['effect_obs', 'boot_ci_low', 'boot_ci_high', 'p_boot_two_sided']].to_string(index=False))




ValueError: invalid literal for int() with base 10: 'spx'